In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
import matplotlib.pyplot as plt

In [3]:
%matplotlib inline

In [4]:
# Notebook nằm trong Project/code_notebook/
# Dữ liệu nằm trong Project/data/lastfm-dataset-1K/
dirname = os.path.join(os.path.dirname(os.getcwd()), 'data', 'lastfm-dataset-1K')

In [5]:
filepath = os.path.join(dirname, 'userid-timestamp-artid-artname-traid-traname.tsv')

In [6]:
%%time

df = pd.read_csv(
    filepath, sep='\t', header=None,
    names=[
        'user_id', 'timestamp', 'artist_id', 'artist_name', 'track_id', 'track_name'
    ],
    skiprows=[
        2120260-1, 2446318-1, 11141081-1,
        11152099-1, 11152402-1, 11882087-1,
        12902539-1, 12935044-1, 17589539-1
    ]
)


CPU times: total: 1min 47s
Wall time: 2min


In [7]:
df["timestamp"] = pd.to_datetime(df.timestamp)
df.sort_values(['user_id', 'timestamp'], ascending=True, inplace=True)
print(f'Number of Records: {len(df):,}\nUnique Users: {df.user_id.nunique()}\nUnique Artist:{df.artist_id.nunique():,}')
df.head(5) 

Number of Records: 19,098,853
Unique Users: 992
Unique Artist:107,295


,user_id,timestamp,artist_id,artist_name,track_id,track_name
16684,user_000001,2006-08-13 13:59:20+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,c4633ab1-e715-477f-8685-afa5f2058e42,The Launching Of Big Face
16683,user_000001,2006-08-13 14:03:29+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,bc2765af-208c-44c5-b3b0-cf597a646660,Zn Zero
16682,user_000001,2006-08-13 14:10:43+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,aa9c5a80-5cbe-42aa-a966-eb3cfa37d832,The Return Of Super Barrio - End Credits
16681,user_000001,2006-08-13 14:17:40+00:00,67fb65b5-6589-47f0-9371-8a40eb268dfb,Tommy Guerrero,d9b1c1da-7e47-4f97-a135-77260f2f559d,Mission Flats
16680,user_000001,2006-08-13 14:19:06+00:00,1cfbc7d1-299c-46e6-ba4c-1facb84ba435,Artful Dodger,120bb01c-03e4-465f-94a0-dce5e9fac711,What You Gonna Do?


In [8]:
df.info()

<class 'pandas.DataFrame'>
Index: 19098853 entries, 16684 to 19080480
Data columns (total 6 columns):
 #   Column       Dtype              
---  ------       -----              
 0   user_id      str                
 1   timestamp    datetime64[us, UTC]
 2   artist_id    str                
 3   artist_name  str                
 4   track_id     str                
 5   track_name   str                
dtypes: datetime64[us, UTC](1), str(5)
memory usage: 2.9 GB


In [9]:
save_filepath = os.path.join(dirname, 'lastfm-dataset-1k.snappy.parquet')

In [10]:
df.to_parquet(save_filepath, compression='snappy', index=False)

In [11]:
del df

In [12]:
%%time

df = pd.read_parquet(save_filepath)
print(f'Number of Records: {len(df):,}\nUnique Users: {df.user_id.nunique()}\nUnique Artist:{df.artist_id.nunique():,}')
df.head(5)

Number of Records: 19,098,853
Unique Users: 992
Unique Artist:107,295
CPU times: total: 20.4 s
Wall time: 43.3 s


,user_id,timestamp,artist_id,artist_name,track_id,track_name
0,user_000001,2006-08-13 13:59:20+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,c4633ab1-e715-477f-8685-afa5f2058e42,The Launching Of Big Face
1,user_000001,2006-08-13 14:03:29+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,bc2765af-208c-44c5-b3b0-cf597a646660,Zn Zero
2,user_000001,2006-08-13 14:10:43+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,aa9c5a80-5cbe-42aa-a966-eb3cfa37d832,The Return Of Super Barrio - End Credits
3,user_000001,2006-08-13 14:17:40+00:00,67fb65b5-6589-47f0-9371-8a40eb268dfb,Tommy Guerrero,d9b1c1da-7e47-4f97-a135-77260f2f559d,Mission Flats
4,user_000001,2006-08-13 14:19:06+00:00,1cfbc7d1-299c-46e6-ba4c-1facb84ba435,Artful Dodger,120bb01c-03e4-465f-94a0-dce5e9fac711,What You Gonna Do?


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19098853 entries, 0 to 19098852
Data columns (total 6 columns):
 #   Column       Dtype              
---  ------       -----              
 0   user_id      str                
 1   timestamp    datetime64[us, UTC]
 2   artist_id    str                
 3   artist_name  str                
 4   track_id     str                
 5   track_name   str                
dtypes: datetime64[us, UTC](1), str(5)
memory usage: 2.7 GB


### Generate 100 User Subset

In [14]:
rand_user_ids = df.user_id.unique()[::20]
print(f'Number of Unique Users: {len(rand_user_ids)}')

Number of Unique Users: 50


In [15]:
%%time

small_df = df.loc[df.user_id.isin(rand_user_ids)].sort_values(['user_id', 'timestamp'])
print(f'Number of Records: {len(small_df):,}\nUnique Users: {small_df.user_id.nunique()}\nUnique Artist:{small_df.artist_id.nunique():,}')

Number of Records: 776,336
Unique Users: 50
Unique Artist:16,774
CPU times: total: 8.06 s
Wall time: 15 s


In [16]:
small_df.head(2)

,user_id,timestamp,artist_id,artist_name,track_id,track_name
0,user_000001,2006-08-13 13:59:20+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,c4633ab1-e715-477f-8685-afa5f2058e42,The Launching Of Big Face
1,user_000001,2006-08-13 14:03:29+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,bc2765af-208c-44c5-b3b0-cf597a646660,Zn Zero


In [17]:
save_filepath = os.path.join(dirname, 'lastfm-dataset-50.snappy.parquet')

In [18]:
small_df.to_parquet(save_filepath, compression='snappy', index=False)

In [19]:
del small_df

In [20]:
%%time

small_df = pd.read_parquet(save_filepath)
print(f'Number of Records: {len(small_df):,}\nUnique Users: {small_df.user_id.nunique()}\nUnique Artist:{small_df.artist_id.nunique():,}')
small_df.head(5)

Number of Records: 776,336
Unique Users: 50
Unique Artist:16,774
CPU times: total: 500 ms
Wall time: 330 ms


,user_id,timestamp,artist_id,artist_name,track_id,track_name
0,user_000001,2006-08-13 13:59:20+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,c4633ab1-e715-477f-8685-afa5f2058e42,The Launching Of Big Face
1,user_000001,2006-08-13 14:03:29+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,bc2765af-208c-44c5-b3b0-cf597a646660,Zn Zero
2,user_000001,2006-08-13 14:10:43+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,aa9c5a80-5cbe-42aa-a966-eb3cfa37d832,The Return Of Super Barrio - End Credits
3,user_000001,2006-08-13 14:17:40+00:00,67fb65b5-6589-47f0-9371-8a40eb268dfb,Tommy Guerrero,d9b1c1da-7e47-4f97-a135-77260f2f559d,Mission Flats
4,user_000001,2006-08-13 14:19:06+00:00,1cfbc7d1-299c-46e6-ba4c-1facb84ba435,Artful Dodger,120bb01c-03e4-465f-94a0-dce5e9fac711,What You Gonna Do?


### Compute Play_count (user × track)

In [21]:
%%time

# Load full dataset
full_path = os.path.join(dirname, 'lastfm-dataset-1k.snappy.parquet')
df = pd.read_parquet(full_path)
print(f'Loaded: {len(df):,} records, {df.user_id.nunique()} users')

Loaded: 19,098,853 records, 992 users
CPU times: total: 16.4 s
Wall time: 8.79 s


In [22]:
%%time

# Tính số lần mỗi user nghe mỗi track
play_count_df = (
    df.groupby(['user_id', 'track_id', 'track_name', 'artist_id', 'artist_name'])
    .size()
    .reset_index(name='Play_count')
)
play_count_df['log_Play_count'] = np.log1p(play_count_df['Play_count'])
print(f'Shape: {play_count_df.shape}')
print(f'Unique users: {play_count_df.user_id.nunique()}')
print(f'Play_count stats:')
print(play_count_df['Play_count'].describe())
play_count_df.head(5)

Shape: (3957927, 7)
Unique users: 992
Play_count stats:
count    3.957927e+06
mean     4.278996e+00
std      1.056516e+01
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      4.000000e+00
max      2.069000e+03
Name: Play_count, dtype: float64
CPU times: total: 47.7 s
Wall time: 53.4 s


,user_id,track_id,track_name,artist_id,artist_name,Play_count,log_Play_count
0,user_000001,00237585-8e04-4cd5-a785-f2185492ab0b,Monday Nightcap,7e66c9f6-2451-408e-88ff-2af55276caf1,Opiate,2,1.098612
1,user_000001,0024d72c-136f-49f2-9078-ce4b39b94d3f,Something In The Way,3d05eb8b-1644-4143-9a61-b28e33c4d85f,4Hero,4,1.609438
2,user_000001,0025055f-39c3-43e2-b874-3bf42bbc9212,Squeeze Me (Feat. Ben Westbeech),472fcf2f-eaa2-44a2-8e8f-2ecd23cd0fc7,Kraak & Smaak,2,1.098612
3,user_000001,002e254d-4624-49f4-b78a-b40711b9e4f3,Tak 4,7e54d133-2525-4bc0-ae94-65584145a386,Plaid,3,1.386294
4,user_000001,00b07689-ec4c-4773-94ce-06f3d198431e,Wanderlust,87c5dedd-371d-4a53-9f7f-80522fb7f3cb,Björk,4,1.609438


In [23]:
# Lưu play_count full
save_play_count_path = os.path.join(dirname, 'play_count_full.snappy.parquet')
play_count_df.to_parquet(save_play_count_path, compression='snappy', index=False)
print(f'Saved to {save_play_count_path}')

Saved to C:\Users\bichn\OneDrive\DocumentCloud\Tai_lieu_cac_mon_hoc\DAA\Project\data\lastfm-dataset-1K\play_count_full.snappy.parquet


### Generate Samples (diverse users, ≥50 users each)

Lấy sample theo cách shuffle danh sách user rồi slice — đảm bảo mỗi sample có đủ độ đa dạng user.

In [24]:
# Tạo folder samples nếu chưa có
samples_dir = os.path.join(os.path.dirname(dirname), 'samples')
os.makedirs(samples_dir, exist_ok=True)
print(f'Samples folder: {samples_dir}')

Samples folder: C:\Users\bichn\OneDrive\DocumentCloud\Tai_lieu_cac_mon_hoc\DAA\Project\data\samples


In [25]:
import random

all_users = play_count_df['user_id'].unique()
print(f'Total unique users: {len(all_users)}')

# Shuffle để đảm bảo đa dạng
random.seed(42)
shuffled_users = all_users.copy()
random.shuffle(shuffled_users)

# Các kích cỡ sample: 50, 100, 200, 500, toàn bộ
sample_sizes = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]
sample_sizes = [s for s in sample_sizes if s <= len(all_users)]  # bỏ size > tổng user
print(f'Sample sizes to generate: {sample_sizes}')

Total unique users: 992
Sample sizes to generate: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]


In [26]:
%%time

for n_users in sample_sizes:
    selected_users = shuffled_users[:n_users]
    sample = play_count_df[play_count_df['user_id'].isin(selected_users)].copy()
    
    fname = f'play_count_sample_{n_users}users.snappy.parquet'
    fpath = os.path.join(samples_dir, fname)
    sample.to_parquet(fpath, compression='snappy', index=False)
    
    print(f'[{n_users} users] records={len(sample):,} | '
          f'unique tracks={sample.track_id.nunique():,} | '
          f'saved → {fname}')

[50 users] records=175,484 | unique tracks=119,248 | saved → play_count_sample_50users.snappy.parquet
[100 users] records=385,720 | unique tracks=218,164 | saved → play_count_sample_100users.snappy.parquet
[150 users] records=563,417 | unique tracks=291,625 | saved → play_count_sample_150users.snappy.parquet
[200 users] records=857,756 | unique tracks=386,338 | saved → play_count_sample_200users.snappy.parquet
[250 users] records=1,045,174 | unique tracks=434,317 | saved → play_count_sample_250users.snappy.parquet
[300 users] records=1,243,964 | unique tracks=491,540 | saved → play_count_sample_300users.snappy.parquet
[350 users] records=1,416,014 | unique tracks=531,691 | saved → play_count_sample_350users.snappy.parquet
[400 users] records=1,606,599 | unique tracks=569,948 | saved → play_count_sample_400users.snappy.parquet
[450 users] records=1,797,024 | unique tracks=612,392 | saved → play_count_sample_450users.snappy.parquet
[500 users] records=1,991,538 | unique tracks=652,782 | 

In [27]:
# Kiểm tra nhanh một sample
check = pd.read_parquet(os.path.join(samples_dir, 'play_count_sample_100users.snappy.parquet'))
print(f'Sample 100 users — shape: {check.shape}')
print(check[['user_id','track_name','Play_count','log_Play_count']].head(10))

Sample 100 users — shape: (385720, 7)
       user_id               track_name  Play_count  log_Play_count
0  user_000002               Ný Batterí           7        2.079442
1  user_000002              1000 Fragen           3        1.386294
2  user_000002        Universal Soldier           1        0.693147
3  user_000002                  Kreisen           3        1.386294
4  user_000002                  Love Me           3        1.386294
5  user_000002         The Wake Up Song           1        0.693147
6  user_000002                   Run On           1        0.693147
7  user_000002  Other Side Of The World          15        2.772589
8  user_000002   The Start Of Something          33        3.526361
9  user_000002              Another Day           3        1.386294
